# REDES NEURAIS ARTIFICIAIS
## PARTE 1: Classificação Binária — Diagnóstico de Câncer de Mama

Base de dados: [Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic)

In [1]:
import tensorflow as tf
from scikeras.wrappers import KerasClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV
from tensorflow.keras import backend as k
from tensorflow.keras.models import Sequential

## 1. Carregar os dados

Carregue a base de dados e faça a divisão de treino e teste com `test_size=0.25`.

In [2]:
def carregar_dados():
    dataset = load_breast_cancer()
    X = dataset.data
    y = dataset.target
    return X, y


def separar_dados(X, y):
    X_treinamento, X_teste, y_treinamento, y_teste = train_test_split(
        X, y, test_size=0.25, random_state=0
    )
    return X_treinamento, X_teste, y_treinamento, y_teste


X, y = carregar_dados()
X_treinamento, X_teste, y_treinamento, y_teste = separar_dados(X, y)

print(X_treinamento.shape, y_treinamento.shape)
print(X_teste.shape, y_teste.shape)

(426, 30) (426,)
(143, 30) (143,)


## 2 e 3. Estrutura da Rede Neural e Sequential

Crie a RNA com as seguintes configurações:

- **a)** Camada de entrada com **30 neurônios**
- **b)** Camada oculta densa com **16 neurônios** — valor entre entrada (30) e saída (1), potência de 2 para eficiência computacional; equilibra capacidade e risco de overfitting
- **c)** Ativação `relu` e inicializador `random_uniform`
- **d)** Camada de saída com `sigmoid` — mapeia a saída para [0,1], interpretável como probabilidade para classificação binária (maligno/benigno)

**3.** `Sequential` é utilizado pois a RNA é uma pilha linear de camadas, onde cada camada tem exatamente uma entrada e uma saída.

In [3]:
def criar_rede_neural():
    rede_neural = Sequential([
        tf.keras.layers.InputLayer(shape=(30,)),
        tf.keras.layers.Dense(units=16, activation="relu", kernel_initializer="random_uniform"),
        tf.keras.layers.Dense(units=1, activation="sigmoid"),
    ])
    return rede_neural


rede_neural = criar_rede_neural()

## 4. Summary da RNA

O `summary()` exibe para cada camada:

- **Layer**: nome e tipo da camada
- **Output Shape**: `(None, N)` — `None` é o batch size dinâmico, `N` é o número de neurônios
- **Param #**: total de pesos + biases treináveis
  - Camada oculta: 30×16 + 16 biases = **496 parâmetros**
  - Camada de saída: 16×1 + 1 bias = **17 parâmetros**
  - **Total: 513 parâmetros treináveis**

In [4]:
rede_neural.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 513 (2.00 KB)

 Trainable params: 513 (2.00 KB)

 Non-trainable params: 0 (0.00 B)

## 5 e 6. Compilar — Adam, Binary Crossentropy, Binary Accuracy

**5.** Adiciona otimizador Adam, loss `binary_crossentropy` e métrica `binary_accuracy`.

**6.** Otimizadores ajustam os pesos para minimizar a função de perda. O **Adam** combina:
- **Momentum**: média móvel dos gradientes para acelerar direções consistentes
- **RMSprop**: taxa de aprendizado adaptativa por parâmetro

Converge de forma eficiente sem precisar ajustar manualmente a learning rate.

In [5]:
def compilar_rede_neural(rede_neural):
    rede_neural.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss="binary_crossentropy",
        metrics=["binary_accuracy"],
    )
    return rede_neural


rede_neural = compilar_rede_neural(rede_neural)

## 7 e 8. Treinamento

**7.** Com 426 amostras e `batch_size=10`, são usados ⌈426/10⌉ = **43 batches por época**.

**8.** O treinamento ocorre por **100 épocas**, totalizando **4.300 atualizações de pesos**.

In [6]:
def treinar_rede_neural(rede_neural, X_treinamento, y_treinamento):
    return rede_neural.fit(X_treinamento, y_treinamento, batch_size=10, epochs=100)


treinar_rede_neural(rede_neural, X_treinamento, y_treinamento)

Epoch 1/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - binary_accuracy: 0.5587 - loss: 1.5275   
Epoch 2/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.8615 - loss: 0.4112
Epoch 3/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.8709 - loss: 0.3606
Epoch 4/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.8638 - loss: 0.3379
Epoch 5/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.8920 - loss: 0.2809
Epoch 6/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - binary_accuracy: 0.8920 - loss: 0.2602
Epoch 7/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - binary_accuracy: 0.9014 - loss: 0.2562
Epoch 8/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9131 - loss: 0.2434
Epoch 9/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9108 - loss: 0.2480
Epoch 10/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9061 - loss: 0.2261
Epoch 11/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.910

## 9. Previsões

O resultado é um valor entre 0 e 1 porque a camada de saída usa `sigmoid`, que mapeia qualquer valor real para [0,1], representando a **probabilidade** de a amostra ser da classe positiva (maligno).

In [7]:
previsoes = rede_neural.predict(X_teste)
previsoes

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


array([[1.34758309e-01],
       [9.91337359e-01],
       [9.98909593e-01],
       [9.47749317e-01],
       [9.99826670e-01],
       [9.96647298e-01],
       [9.99122262e-01],
       [9.98982728e-01],
       [9.78240550e-01],
       [9.99554694e-01],
       [9.59623694e-01],
       [9.50371802e-01],
       [9.98722315e-01],
       [8.25433433e-01],
       [9.58998263e-01],
       [2.18430348e-02],
       [9.89589155e-01],
       [2.01650039e-07],
       [2.06524223e-01],
       [3.48598261e-09],
       [1.20330369e-02],
       [6.72875345e-01],
       [9.98062551e-01],
       [9.92720604e-01],
       [4.89412397e-01],
       [9.90941525e-01],
       [9.97632146e-01],
       [9.55238402e-01],
       [9.97334778e-01],
       [9.62648699e-08],
       [9.99544680e-01],
       [1.21518212e-07],
       [9.43245828e-01],
       [8.49332660e-03],
       [9.99566138e-01],
       [1.49134090e-02],
       [9.73218203e-01],
       [8.65250549e-05],
       [9.96505857e-01],
       [3.55696143e-03],


## 10. Converter para Binário

Aplica threshold de 0.5: probabilidade ≥ 0.5 → `True` (maligno), < 0.5 → `False` (benigno).

In [8]:
previsoes = previsoes > 0.5
previsoes

array([[False],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [False],
       [ True],
       [False],
       [False],
       [False],
       [False],
       [ True],
       [ True],
       [ True],
       [False],
       [ True],
       [ True],
       [ True],
       [ True],
       [False],
       [ True],
       [False],
       [ True],
       [False],
       [ True],
       [False],
       [ True],
       [False],
       [ True],
       [False],
       [False],
       [ True],
       [False],
       [ True],
       [ True],
       [False],
       [ True],
       [ True],
       [ True],
       [False],
       [False],
       [ True],
       [False],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [ True],
       [False],
       [False],
       [False],
       [

## 9. Avaliar o resultado da RNA

O `evaluate()` retorna a **loss** e a **binary_accuracy** nos dados de teste — indicando o desempenho real da rede em amostras que ela nunca viu durante o treinamento.

In [9]:
rede_neural.evaluate(X_teste, y_teste)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - binary_accuracy: 0.9510 - loss: 0.1518  


[0.1517971009016037, 0.9510489702224731]

---
## Camadas e Otimização da RNA

### 10. Segunda camada oculta

Adiciona mais uma camada densa com 16 neurônios, ativação `relu` e inicializador `random_uniform`.

Total de parâmetros com 2 camadas ocultas:
- Dense1: 30×16 + 16 biases = 496
- Dense2: 16×16 + 16 biases = 272
- Saída: 16×1 + 1 bias = 17
- **Total: 785 parâmetros treináveis**

In [10]:
def criar_rede_neural_duas_camadas():
    rede_neural = Sequential([
        tf.keras.layers.InputLayer(shape=(30,)),
        tf.keras.layers.Dense(units=16, activation="relu", kernel_initializer="random_uniform"),
        tf.keras.layers.Dense(units=16, activation="relu", kernel_initializer="random_uniform"),
        tf.keras.layers.Dense(units=1, activation="sigmoid"),
    ])
    return rede_neural


rede_neural_2 = criar_rede_neural_duas_camadas()
rede_neural_2.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 16)             │           496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 785 (3.07 KB)

 Trainable params: 785 (3.07 KB)

 Non-trainable params: 0 (0.00 B)

### 11. Otimizador com learning_rate e clipvalue

- **`learning_rate=0.001`**: controla o tamanho do passo na atualização dos pesos — muito alto diverge, muito baixo aprende devagar.
- **`clipvalue=0.5`**: limita os gradientes ao intervalo [-0.5, 0.5] para evitar *exploding gradients*.

In [11]:
def compilar_rede_neural_otimizada(rede_neural):
    otimizador = tf.keras.optimizers.Adam(learning_rate=0.001, clipvalue=0.5)
    rede_neural.compile(
        optimizer=otimizador,
        loss="binary_crossentropy",
        metrics=["binary_accuracy"],
    )
    return rede_neural


rede_neural_2 = compilar_rede_neural_otimizada(rede_neural_2)

### 12. Treinar e avaliar a RNA com 2 camadas

Aumentar a quantidade de camadas aumenta a capacidade da rede, mas em datasets pequenos pode causar **overfitting** (boa acurácia no treino, pior no teste) ou não trazer ganho significativo.

In [12]:
treinar_rede_neural(rede_neural_2, X_treinamento, y_treinamento)
rede_neural_2.predict(X_teste)
rede_neural_2.evaluate(X_teste, y_teste)

Epoch 1/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - binary_accuracy: 0.5305 - loss: 1.1192   
Epoch 2/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.8216 - loss: 0.4731
Epoch 3/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.8873 - loss: 0.3365
Epoch 4/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.8803 - loss: 0.3160
Epoch 5/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - binary_accuracy: 0.8944 - loss: 0.2749
Epoch 6/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9061 - loss: 0.2410
Epoch 7/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.8826 - loss: 0.2776 
Epoch 8/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9014 - loss: 0.2934
Epoch 9/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - binary_accuracy: 0.9038 - loss: 0.2493
Epoch 10/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - binary_accuracy: 0.8944 - loss: 0.2856
Epoch 11/100
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - binary_accuracy: 0.88

[0.24933911859989166, 0.9090909361839294]

---
## K-Fold Cross Validation

### 13 e 14. K-Fold com 10 folds

**13.** O K-Fold divide o dataset em k=10 partes (folds). A cada iteração, 1 fold é usado como teste e os 9 restantes como treino, repetindo até que todos os folds sirvam de teste. O resultado final é a média das 10 acurácias, reduzindo o viés de uma única divisão treino/teste. O `k.clear_session()` limpa os pesos da sessão anterior a cada fold.

**14.** Desvio padrão baixo → modelo estável entre os folds (pouca variância). Desvio padrão alto → acurácia dependente do fold — sinal de instabilidade.

In [13]:
def criar_rede_kfold():
    k.clear_session()
    rede_neural = Sequential([
        tf.keras.layers.InputLayer(shape=(30,)),
        tf.keras.layers.Dense(units=16, activation="relu", kernel_initializer="random_uniform"),
        tf.keras.layers.Dense(units=16, activation="relu", kernel_initializer="random_uniform"),
        tf.keras.layers.Dense(units=1, activation="sigmoid"),
    ])
    otimizador = tf.keras.optimizers.Adam(learning_rate=0.001, clipvalue=0.5)
    rede_neural.compile(optimizer=otimizador, loss="binary_crossentropy", metrics=["binary_accuracy"])
    return rede_neural


classificador = KerasClassifier(model=criar_rede_kfold, epochs=100, batch_size=10)
resultados = cross_val_score(estimator=classificador, X=X, y=y, cv=10, scoring="accuracy")

print(f"Acurácia média: {resultados.mean():.4f}")
print(f"Desvio padrão:  {resultados.std():.4f}")
resultados


Epoch 1/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - binary_accuracy: 0.7773 - loss: 0.6016  
Epoch 2/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.8828 - loss: 0.3988
Epoch 3/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9023 - loss: 0.3032
Epoch 4/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9082 - loss: 0.2427
Epoch 5/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9062 - loss: 0.2490
Epoch 6/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9082 - loss: 0.2833
Epoch 7/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9043 - loss: 0.2213
Epoch 8/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9277 - loss: 0.2014
Epoch 9/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - binary_accuracy: 0.9141 - loss: 0.2240
Epoch 10/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.9043 - loss: 0.2448
Epoch 11/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.904

array([0.96491228, 0.89473684, 0.96491228, 0.89473684, 0.9122807 ,
       0.92982456, 0.94736842, 0.94736842, 0.9122807 , 0.89285714])

---
## Overfitting e Dropout

### 14. Dropout 20% nas camadas ocultas

O Dropout desativa aleatoriamente 20% dos neurônios a cada passo de treino, forçando a rede a aprender representações redundantes e reduzindo a dependência de neurônios específicos — principal técnica contra overfitting.

**Esperado**: acurácia média semelhante ou ligeiramente menor, mas desvio padrão menor (modelo mais estável/generalizável).

In [ ]:
def criar_rede_com_dropout():
    k.clear_session()
    rede_neural = Sequential([
        tf.keras.layers.InputLayer(shape=(30,)),
        tf.keras.layers.Dense(units=16, activation="relu", kernel_initializer="random_uniform"),
        tf.keras.layers.Dropout(rate=0.2),
        tf.keras.layers.Dense(units=16, activation="relu", kernel_initializer="random_uniform"),
        tf.keras.layers.Dropout(rate=0.2),
        tf.keras.layers.Dense(units=1, activation="sigmoid"),
    ])
    otimizador = tf.keras.optimizers.Adam(learning_rate=0.001, clipvalue=0.5)
    rede_neural.compile(optimizer=otimizador, loss="binary_crossentropy", metrics=["binary_accuracy"])
    return rede_neural


classificador_dropout = KerasClassifier(model=criar_rede_com_dropout, epochs=100, batch_size=10)
resultados_dropout = cross_val_score(estimator=classificador_dropout, X=X, y=y, cv=10, scoring="accuracy")

print(f"Acurácia média (dropout): {resultados_dropout.mean():.4f}")
print(f"Desvio padrão  (dropout): {resultados_dropout.std():.4f}")
resultados_dropout

Epoch 1/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - binary_accuracy: 0.5449 - loss: 0.8134
Epoch 2/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.7520 - loss: 0.5845
Epoch 3/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.8164 - loss: 0.4545
Epoch 4/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - binary_accuracy: 0.8496 - loss: 0.3956
Epoch 5/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.8672 - loss: 0.3512
Epoch 6/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.8789 - loss: 0.3050
Epoch 7/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - binary_accuracy: 0.8848 - loss: 0.3171
Epoch 8/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.8809 - loss: 0.3385
Epoch 9/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.8945 - loss: 0.3059
Epoch 10/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.8926 - loss: 0.3196
Epoch 11/100
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - binary_accuracy: 0.8828 -

---
## Tuning dos Hiperparâmetros

### 15 e 16. GridSearchCV

**15.** A RNA foi configurada para receber hiperparâmetros como argumentos, permitindo que o `GridSearchCV` substitua cada combinação automaticamente. O `KerasClassifier` adapta o modelo Keras à interface do sklearn, e o `GridSearchCV` testa todas as combinações do `param_grid` com validação cruzada (`cv=5`), retornando a combinação de maior acurácia média.

**16.** Sim, é possível melhorar a acurácia ampliando o espaço de busca (mais otimizadores, learning rates, neurônios e camadas), usando `RandomizedSearchCV` para explorar combinações aleatórias de forma mais eficiente, ou aplicando `BatchNormalization` e early stopping.

In [ ]:
def criar_rede_tuning(optimizer, loss, kernel_initializer, activation, neurons):
    k.clear_session()
    rede_neural = Sequential([
        tf.keras.layers.InputLayer(shape=(30,)),
        tf.keras.layers.Dense(units=neurons, activation=activation, kernel_initializer=kernel_initializer),
        tf.keras.layers.Dropout(rate=0.2),
        tf.keras.layers.Dense(units=neurons, activation=activation, kernel_initializer=kernel_initializer),
        tf.keras.layers.Dropout(rate=0.2),
        tf.keras.layers.Dense(units=1, activation="sigmoid"),
    ])
    rede_neural.compile(optimizer=optimizer, loss=loss, metrics=["binary_accuracy"])
    return rede_neural


classificador_tuning = KerasClassifier(model=criar_rede_tuning)
parametros = {
    "batch_size": [10, 30],
    "epochs": [50],
    "model__optimizer": ["adam"],
    "model__loss": ["binary_crossentropy"],
    "model__kernel_initializer": ["random_uniform", "normal"],
    "model__activation": ["relu"],
    "model__neurons": [16],
}

grid_search = GridSearchCV(estimator=classificador_tuning, param_grid=parametros, scoring="accuracy", cv=5)
grid_search = grid_search.fit(X, y)

print(f"Melhores parâmetros: {grid_search.best_params_}")
print(f"Melhor acurácia:     {grid_search.best_score_:.4f}")